# Peanut Drought ML — Colab quick start

Run each cell in order in a fresh Python runtime. Upload the complete **package ZIP**, not an individual file. This walkthrough runs a synthetic demonstration, not a biological validation. No GPU is required.

The package accepts processed numeric expression data; it does not process raw RNA-seq counts. See README.md before research use.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, tempfile, subprocess, sys, json

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith(".zip")]
if len(archives) != 1:
    raise ValueError("Upload exactly one complete Peanut Drought ML package ZIP.")
workspace = Path(tempfile.mkdtemp(prefix="peanutml_"))
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    for entry in archive.infolist():
        target = (workspace / entry.filename).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError("Invalid archive path")
    archive.extractall(workspace)
projects = [p.parent for p in workspace.rglob("pyproject.toml")
            if (p.parent / "src" / "peanutml").is_dir()]
if len(projects) != 1:
    raise ValueError("Expected one package project in the uploaded archive.")
project = projects[0]
print("Package folder:", project)
print("Python:", sys.version)

## Install

This installs the uploaded local release and its dependencies. The package is not being downloaded from PyPI. If Colab requests a session restart after dependency changes, restart and rerun the cells from the top.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", str(project)], check=True)
subprocess.run([sys.executable, "-c", "import peanutml; print('Installed Peanut Drought ML:', peanutml.__version__)"], check=True)

## Run the demonstration

The demo uses simulated expression data and a small search. Research defaults use much larger searches. A successful demo confirms that the basic workflow runs, not that a model will generalize to new experiments.

In [ ]:
subprocess.run([sys.executable, str(project / "examples" / "demo.py")], cwd=project, check=True)

## Inspect the output files

In [ ]:
results = project / "demo_results"
for path in sorted(results.rglob("*")):
    if path.is_file():
        print(path.relative_to(results))

## View available commands

In [ ]:
import shutil
cli = shutil.which("peanutml")
if cli is None:
    raise RuntimeError("peanutml command not found after installation")
subprocess.run([cli, "--help"], check=True)

## Download the demonstration results

Download before ending the runtime. These files are synthetic outputs, not research findings.

In [ ]:
environment = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(results / "environment.txt").write_text(environment)
archive_path = shutil.make_archive(str(workspace / "PeanutML_Demo_Results"), "zip", results)
files.download(archive_path)

## Your research analysis

Follow README.md for `peanutml cv`, `peanutml train`, and `peanutml validate`. Input rows must be samples and gene columns must be appropriately transformed numeric expression values. Remove unrelated metadata or explicitly select gene columns through the API.

The included research CSVs are in `provenance/data/`. Choose fixed versus tuned training explicitly. No trained classifier is supplied as a general-purpose default.

Upload/download reference: https://colab.research.google.com/notebooks/io.ipynb